In [3]:
import numpy as np
import pandas as pd
import requests
import os
import json
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv('../.env')

api_key = os.getenv('OPENWEATHER_API_KEY')
print('Libraries installed!!!')

Libraries installed!!!


In [4]:
# Download Zomato dataset from a reliable source
url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/zomato.csv"

try:
    print("Trying source 1...")
    zomato = pd.read_csv(url)
    print(f"Success! Shape: {zomato.shape}")
except:
    try:
        print("Trying source 2...")
        url2 = "https://raw.githubusercontent.com/FlipRoboTechnologies/ML_-Data-Sets/main/Zomato/zomato.csv"
        zomato = pd.read_csv(url2, encoding='latin-1')
        print(f"Success! Shape: {zomato.shape}")
    except:
        print("Downloading from backup source...")
        # Create a realistic synthetic Zomato-like dataset
        np.random.seed(42)
        cuisines = ['North Indian', 'Chinese', 'Fast Food', 'South Indian', 
                   'Pizza', 'Biryani', 'Mughlai', 'Desserts', 'Beverages']
        zones = ['Connaught Place', 'Hauz Khas', 'Rohini', 
                 'Laxmi Nagar', 'Rajouri Garden', 'Noida']
        
        zomato = pd.DataFrame({
            'restaurant_name': [f'Restaurant_{i}' for i in range(1000)],
            'cuisine': np.random.choice(cuisines, 1000),
            'location': np.random.choice(zones, 1000),
            'avg_order_value': np.random.randint(150, 800, 1000),
            'rating': np.round(np.random.uniform(3.0, 4.9, 1000), 1),
            'delivery_time_mins': np.random.randint(20, 60, 1000),
            'monthly_orders': np.random.randint(100, 2000, 1000)
        })
        print(f"Created synthetic restaurant data. Shape: {zomato.shape}")

print(f"\nColumns: {zomato.columns.tolist()}")
print(zomato.head(3))

zomato.to_csv("../data/raw/zomato_raw.csv", index=False)
print("\nSaved to data/raw/zomato_raw.csv")

Trying source 1...
Trying source 2...
Created synthetic restaurant data. Shape: (1000, 7)

Columns: ['restaurant_name', 'cuisine', 'location', 'avg_order_value', 'rating', 'delivery_time_mins', 'monthly_orders']
  restaurant_name       cuisine        location  avg_order_value  rating  \
0    Restaurant_0       Mughlai          Rohini              239     4.4   
1    Restaurant_1  South Indian  Rajouri Garden              494     4.6   
2    Restaurant_2      Desserts           Noida              499     3.1   

   delivery_time_mins  monthly_orders  
0                  38            1227  
1                  48            1878  
2                  26            1585  

Saved to data/raw/zomato_raw.csv


In [5]:
# 6 Delhi delivery zones based on real neighborhoods
delhi_zones = {
    'zone_1': {
        'name': 'Connaught Place / Central Delhi',
        'lat': 28.6315, 'lon': 77.2167,
        'type': 'commercial',
        'base_demand': 120  # orders per 30-min window at peak
    },
    'zone_2': {
        'name': 'South Delhi / Hauz Khas',
        'lat': 28.5494, 'lon': 77.2001,
        'type': 'residential_affluent',
        'base_demand': 150
    },
    'zone_3': {
        'name': 'North Delhi / Rohini',
        'lat': 28.7041, 'lon': 77.1025,
        'type': 'residential',
        'base_demand': 90
    },
    'zone_4': {
        'name': 'East Delhi / Laxmi Nagar',
        'lat': 28.6276, 'lon': 77.2772,
        'type': 'residential',
        'base_demand': 85
    },
    'zone_5': {
        'name': 'West Delhi / Rajouri Garden',
        'lat': 28.6475, 'lon': 77.1220,
        'type': 'mixed',
        'base_demand': 100
    },
    'zone_6': {
        'name': 'Noida / Cyber City',
        'lat': 28.5355, 'lon': 77.3910,
        'type': 'tech_hub',
        'base_demand': 130
    }
}

import json
with open('../data/raw/delhi_zones.json', 'w') as f:
    json.dump(delhi_zones, f, indent=2)

print("Delhi zones defined:")
for zone_id, zone in delhi_zones.items():
    print(f"  {zone_id}: {zone['name']} (base demand: {zone['base_demand']} orders/30min)")

Delhi zones defined:
  zone_1: Connaught Place / Central Delhi (base demand: 120 orders/30min)
  zone_2: South Delhi / Hauz Khas (base demand: 150 orders/30min)
  zone_3: North Delhi / Rohini (base demand: 90 orders/30min)
  zone_4: East Delhi / Laxmi Nagar (base demand: 85 orders/30min)
  zone_5: West Delhi / Rajouri Garden (base demand: 100 orders/30min)
  zone_6: Noida / Cyber City (base demand: 130 orders/30min)


In [6]:
# Major Delhi demand events — festivals, IPL, holidays
delhi_events = [
    # Format: (date, event_name, demand_multiplier)
    # demand_multiplier: 1.0 = normal, 2.0 = double demand
    
    # Festivals
    ('2024-10-31', 'Diwali',              3.0),
    ('2024-11-01', 'Diwali Day 2',        2.5),
    ('2024-03-25', 'Holi',               2.0),
    ('2024-01-26', 'Republic Day',        1.8),
    ('2024-08-15', 'Independence Day',    1.8),
    ('2024-10-02', 'Gandhi Jayanti',      1.5),
    ('2024-11-15', 'Guru Nanak Jayanti',  1.6),
    ('2024-12-25', 'Christmas',           1.7),
    ('2024-01-01', 'New Year',            2.5),
    ('2024-12-31', 'New Year Eve',        2.8),
    
    # IPL Season (April-May) — matches drive late night orders
    ('2024-04-01', 'IPL Start',           1.5),
    ('2024-05-26', 'IPL Finals',          2.2),
    
    # Weather events (monsoon = surge)
    ('2024-07-01', 'Monsoon Start Delhi', 2.0),
    ('2024-07-15', 'Heavy Rain Week',     2.5),
    ('2024-08-01', 'Peak Monsoon',        2.3),
    
    # Exam seasons (students order more)
    ('2024-03-01', 'Board Exams Start',   1.3),
    ('2024-05-01', 'JEE/NEET Season',     1.2),
]

events_df = pd.DataFrame(delhi_events, columns=['date', 'event', 'demand_multiplier'])
events_df['date'] = pd.to_datetime(events_df['date'])
events_df.to_csv('../data/raw/delhi_events.csv', index=False)

print(f"Delhi events calendar: {len(events_df)} events")
print(events_df.head(10))

Delhi events calendar: 17 events
        date               event  demand_multiplier
0 2024-10-31              Diwali                3.0
1 2024-11-01        Diwali Day 2                2.5
2 2024-03-25                Holi                2.0
3 2024-01-26        Republic Day                1.8
4 2024-08-15    Independence Day                1.8
5 2024-10-02      Gandhi Jayanti                1.5
6 2024-11-15  Guru Nanak Jayanti                1.6
7 2024-12-25           Christmas                1.7
8 2024-01-01            New Year                2.5
9 2024-12-31        New Year Eve                2.8


In [7]:
# Fetch last 5 days of hourly Delhi weather from Open-Meteo
# (OpenWeatherMap historical data needs paid plan — Open-Meteo is free)

print("Fetching historical Delhi weather...")

url = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=28.6139&longitude=77.2090"
    "&hourly=temperature_2m,relative_humidity_2m,precipitation,"
    "wind_speed_10m,weather_code"
    "&past_days=7"
    "&timezone=Asia/Kolkata"
)

response = requests.get(url)
weather_data = response.json()

weather_df = pd.DataFrame({
    'timestamp':    weather_data['hourly']['time'],
    'temperature':  weather_data['hourly']['temperature_2m'],
    'humidity':     weather_data['hourly']['relative_humidity_2m'],
    'precipitation':weather_data['hourly']['precipitation'],
    'wind_speed':   weather_data['hourly']['wind_speed_10m'],
    'weather_code': weather_data['hourly']['weather_code']
})

weather_df['timestamp'] = pd.to_datetime(weather_df['timestamp'])
weather_df.to_csv('../data/raw/delhi_weather_historical.csv', index=False)

print(f"Historical weather saved: {weather_df.shape}")
print(weather_df.tail(5))

Fetching historical Delhi weather...
Historical weather saved: (336, 6)
              timestamp  temperature  humidity  precipitation  wind_speed  \
331 2026-04-28 19:00:00         34.7        34            0.0         7.2   
332 2026-04-28 20:00:00         33.0        38            0.0         7.6   
333 2026-04-28 21:00:00         31.7        42            0.0         6.8   
334 2026-04-28 22:00:00         30.6        46            0.0         5.8   
335 2026-04-28 23:00:00         29.7        50            0.0         5.1   

     weather_code  
331             3  
332             3  
333             3  
334             3  
335             3  


In [8]:
import numpy as np

np.random.seed(42)

# Generate 90 days of 30-minute interval order data
dates = pd.date_range(
    start='2024-01-01',
    end='2024-03-31',
    freq='30min'
)

records = []

for timestamp in dates:
    hour = timestamp.hour
    day_of_week = timestamp.dayofweek  # 0=Monday, 6=Sunday
    
    for zone_id, zone in delhi_zones.items():
        base = zone['base_demand']
        
        # Time of day multiplier
        if 12 <= hour <= 14:    # Lunch peak
            time_mult = 1.8
        elif 19 <= hour <= 22:  # Dinner peak
            time_mult = 2.2
        elif 7 <= hour <= 9:    # Breakfast
            time_mult = 0.8
        elif 0 <= hour <= 6:    # Late night / early morning
            time_mult = 0.2
        else:
            time_mult = 0.6
        
        # Weekend multiplier
        weekend_mult = 1.3 if day_of_week >= 5 else 1.0
        
        # Zone type multiplier
        if zone['type'] == 'tech_hub' and 12 <= hour <= 14:
            zone_mult = 1.5  # Tech workers order heavily at lunch
        elif zone['type'] == 'residential_affluent':
            zone_mult = 1.2
        else:
            zone_mult = 1.0
        
        # Calculate expected orders with noise
        expected = base * time_mult * weekend_mult * zone_mult
        actual = max(0, int(np.random.normal(expected, expected * 0.15)))
        
        records.append({
            'timestamp': timestamp,
            'zone_id': zone_id,
            'zone_name': zone['name'],
            'orders': actual,
            'hour': hour,
            'day_of_week': day_of_week,
            'is_weekend': day_of_week >= 5
        })

orders_df = pd.DataFrame(records)
orders_df.to_csv('../data/raw/synthetic_orders.csv', index=False)

print(f"Synthetic order data generated: {orders_df.shape}")
print(f"Date range: {orders_df['timestamp'].min()} to {orders_df['timestamp'].max()}")
print(f"\nSample peak hour (7pm, South Delhi):")
sample = orders_df[
    (orders_df['hour'] == 19) & 
    (orders_df['zone_id'] == 'zone_2')
].head(5)
print(sample[['timestamp', 'zone_name', 'orders']])

Synthetic order data generated: (25926, 7)
Date range: 2024-01-01 00:00:00 to 2024-03-31 00:00:00

Sample peak hour (7pm, South Delhi):
              timestamp                zone_name  orders
229 2024-01-01 19:00:00  South Delhi / Hauz Khas     436
235 2024-01-01 19:30:00  South Delhi / Hauz Khas     433
517 2024-01-02 19:00:00  South Delhi / Hauz Khas     337
523 2024-01-02 19:30:00  South Delhi / Hauz Khas     429
805 2024-01-03 19:00:00  South Delhi / Hauz Khas     377


In [9]:
# Verify all files are saved
files = {
    'Zomato raw':           '../data/raw/zomato_raw.csv',
    'Delhi zones':          '../data/raw/delhi_zones.json',
    'Delhi events':         '../data/raw/delhi_events.csv',
    'Historical weather':   '../data/raw/delhi_weather_historical.csv',
    'Synthetic orders':     '../data/raw/synthetic_orders.csv'
}

print("Verifying all data files...\n")
all_good = True

for name, path in files.items():
    if os.path.exists(path):
        if path.endswith('.csv'):
            df = pd.read_csv(path)
            print(f"  {name}: {df.shape[0]:,} rows x {df.shape[1]} cols")
        else:
            print(f"  {name}: saved")
    else:
        print(f"  MISSING: {name}")
        all_good = False

if all_good:
    print("\nAll data files ready!")
    print("Step 2 complete — ready for Step 3!")

Verifying all data files...

  Zomato raw: 1,000 rows x 7 cols
  Delhi zones: saved
  Delhi events: 17 rows x 3 cols
  Historical weather: 336 rows x 6 cols
  Synthetic orders: 25,926 rows x 7 cols

All data files ready!
Step 2 complete — ready for Step 3!
